# TẢI ZIP TN4 — đầy đủ, kèm checkpoint

Notebook **chỉ đọc**. Không train, không xoá, không clone mã.

Lấy **mọi tệp nén của TN4** trên Drive — cả `final.pth` (trọng số model) —
gộp thành một tệp rồi bấm tải về máy.

TN4 gồm bốn cấu hình, mỗi cấu hình ba seed:

| tệp nén Drive | cấu hình |
|---|---|
| `tn4_ds_tcn_c64_k3_..._a0.6_..._seed{0,1,2}` | Ours-64/61 alpha 0,6 |
| `tn4_ds_tcn_c64_k5_..._a0_..._seed{0,1,2}` | Ours-64/121 Pearson thuần ★ |
| `tn4_ds_tcn_c64_k5_..._mse_..._seed{0,1,2}` | Ours-64/121 MSE thuần (nền) |
| `tn4_ds_tcn_c192_k5_..._a0.2_..._seed{0,1,2}` | Ours-192/121 alpha 0,2 |

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Liệt kê mọi tệp nén TN4

In [ ]:
import glob, os, subprocess

SRC = "/content/drive/MyDrive/mobivital"
zips = sorted(p for p in glob.glob(SRC + "/tn4_*.zip"))

tong = 0
print("  %-72s %8s   %s" % ("tệp", "MB", "sửa lúc"))
print("  " + "-" * 100)
for p in zips:
    mb = os.path.getsize(p) / 1048576
    tong += mb
    gio = subprocess.run(["date", "-r", p, "+%m-%d %H:%M"],
                         capture_output=True, text=True).stdout.strip()
    print("  %-72s %8.1f   %s" % (os.path.basename(p)[:72], mb, gio))
print("  " + "-" * 100)
print("  %d tệp  ·  tổng %.0f MB" % (len(zips), tong))

## 3. Gộp NGUYÊN VẸN thành một tệp

Giữ tất cả — `summary.csv`, `scores_*.csv`, `curve.csv`, bảng lựa chọn kênh `.txt`, và `final.pth`. Không bỏ gì.

In [ ]:
import shutil, tarfile, tempfile

work = tempfile.mkdtemp()
out_root = os.path.join(work, "TN4_day_du")
os.makedirs(out_root)

n_pth = n_file = 0
for p in zips:
    ten = os.path.splitext(os.path.basename(p))[0]
    dest = os.path.join(out_root, ten)
    subprocess.run(["unzip", "-oq", p, "-d", dest], check=True)

for r, _, fs in os.walk(out_root):
    for f in fs:
        n_file += 1
        if f.endswith(".pth"):
            n_pth += 1

archive = "/content/TN4_day_du.tar.gz"
with tarfile.open(archive, "w:gz") as tf:
    tf.add(out_root, arcname="TN4_day_du")

mb = os.path.getsize(archive) / 1048576
print("  %d tệp con  ·  %d file final.pth (checkpoint)" % (n_file, n_pth))
print("  -> %s   %.1f MB" % (archive, mb))

## 4. Xem cây thư mục

In [ ]:
print(subprocess.run(["bash", "-lc",
      "cd %s && find TN4_day_du -maxdepth 3 | sort" % work],
      capture_output=True, text=True).stdout)

## 5. Tải về máy

In [ ]:
from google.colab import files
files.download("/content/TN4_day_du.tar.gz")

## 6. Ở máy

```bash
tar -xzf TN4_day_du.tar.gz
```

Mỗi tệp nén một thư mục, mỗi thư mục có `tn4/<run_id>/final.pth`.